In [1]:
import torch

In [3]:
'cuda' if torch.cuda.is_available() else 'cpu'

'cpu'

In [4]:
torch.cuda.device_count()

0

In [14]:
torch.cuda.get_device_properties #it's showing a class because we don't have gpu

<function torch.cuda.get_device_properties(device: torch.device | str | int | None = None) -> torch._utils._CudaDeviceProperties>

In [15]:
torch.cuda.get_device_name #it's showing a class because we don't have gpu

<function torch.cuda.get_device_name(device: torch.device | str | int | None = None) -> str>

In [18]:
torch.cuda.device #it's showing only what we type it's not showing which device we are using because we don't have gpu(cuda)

torch.cuda.device

In [21]:
import numpy as np

arr = np.array([[1,2,3],[4,5,6]])
arr, arr.shape, arr.ndim

(array([[1, 2, 3],
        [4, 5, 6]]),
 (2, 3),
 2)

In [22]:
tensor = torch.Tensor([[1,2,3], [4,5,6]])
tensor, tensor.shape, tensor.ndim

(tensor([[1., 2., 3.],
         [4., 5., 6.]]),
 torch.Size([2, 3]),
 2)

In [23]:
arr * 5, tensor * 5

(array([[ 5, 10, 15],
        [20, 25, 30]]),
 tensor([[ 5., 10., 15.],
         [20., 25., 30.]]))

In [24]:
arr.sum(), tensor.sum()

(np.int64(21), tensor(21.))

In [28]:
tensor_from_arr = torch.from_numpy(arr)
tensor_from_arr, tensor_from_arr.dtype, type(tensor_from_arr), tensor_from_arr.shape

(tensor([[1, 2, 3],
         [4, 5, 6]]),
 torch.int64,
 torch.Tensor,
 torch.Size([2, 3]))

In [34]:
np.ones((2,2)), torch.ones((2,2))

(array([[1., 1.],
        [1., 1.]]),
 tensor([[1., 1.],
         [1., 1.]]))

In [38]:
#tensor.to('cuda') # We don't have cuda hence it's giving an error.

Let's will start pytorch

In [46]:
a = torch.tensor([2., 3.], requires_grad=True)
b = torch.tensor([4., 5.], requires_grad=True)

f = 3 * a**3 - b**2
f

tensor([ 8., 56.], grad_fn=<SubBackward0>)

In [47]:
f.backward(gradient=torch.tensor([1,1]))

In [49]:
a.grad

tensor([36., 81.])

In [51]:
9 * a ** 2

tensor([36., 81.], grad_fn=<MulBackward0>)

In [52]:
b.grad

tensor([ -8., -10.])

In [54]:
 -2 * b

tensor([ -8., -10.], grad_fn=<MulBackward0>)

In [58]:
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [59]:
X, y = load_breast_cancer(return_X_y=True)

In [68]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,  random_state=42)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [74]:
X_train_scaled_tensor = torch.from_numpy(X_train_scaled).float()
X_test_scaled_tensor = torch.from_numpy(X_test_scaled).float()

y_train_tensor = torch.from_numpy(y_train).float().unsqueeze(1)
y_test_tensor = torch.from_numpy(y_test).float().unsqueeze(1)

In [75]:
train_dataset = TensorDataset(X_train_scaled_tensor, y_train_tensor)

In [77]:
X_train_scaled_tensor.shape, y_train_tensor.shape, X_test_scaled_tensor.shape, y_test_tensor.shape

(torch.Size([455, 30]),
 torch.Size([455, 1]),
 torch.Size([114, 30]),
 torch.Size([114, 1]))

In [78]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

In [98]:
class BCNet(nn.Module):
    
    def __init__(self):
        super(BCNet, self).__init__()

        self.fc1 = nn.Linear(30, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 1)

        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.dropout(x)
        x = F.sigmoid(self.fc3(x))

        return x

model = BCNet()

In [99]:
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [100]:
epochs = 20

for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for x_batch, y_batch in train_loader:
        optimizer.zero_grad()
        
        preds = model(x_batch)
        loss = criterion(preds, y_batch)
        
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
    print(f"Epoch: {epoch + 1}: Loss Was {running_loss/len(train_loader):.4f}")

Epoch: 1: Loss Was 0.6718
Epoch: 2: Loss Was 0.5717
Epoch: 3: Loss Was 0.4504
Epoch: 4: Loss Was 0.3161
Epoch: 5: Loss Was 0.2151
Epoch: 6: Loss Was 0.1656
Epoch: 7: Loss Was 0.1151
Epoch: 8: Loss Was 0.1179
Epoch: 9: Loss Was 0.0918
Epoch: 10: Loss Was 0.0843
Epoch: 11: Loss Was 0.0769
Epoch: 12: Loss Was 0.0826
Epoch: 13: Loss Was 0.0658
Epoch: 14: Loss Was 0.0588
Epoch: 15: Loss Was 0.0610
Epoch: 16: Loss Was 0.0535
Epoch: 17: Loss Was 0.0578
Epoch: 18: Loss Was 0.0603
Epoch: 19: Loss Was 0.0446
Epoch: 20: Loss Was 0.0506


In [101]:
with torch.no_grad():
    model.eval()

    preds = model(X_test_scaled_tensor)
    loss = criterion(preds, y_test_tensor).item()

    accuracy = ((preds >= 0.5) == y_test_tensor).float().mean().item()

In [102]:
f"Accuracy: {accuracy:.2f}%"

'Accuracy: 0.98%'